# 🦅 PREDATOR Analytics v61.0-ELITE — CLUSTER DEPLOY ON COLAB

Цей блокнот розгортає **повноцінний Kubernetes кластер (K3d)** всередині Google Colab. 
Він компілює контейнери (Backend + Frontend) так само, як це відбувається на вашому iMac, і розгортає їх через Helm.

**Порядок дій:** Виконайте всі клітинки по черзі.

In [ ]:
%%bash
# 1. ВСТАНОВЛЕННЯ DOCKER, K3D ТА HELM
echo '📦 Встановлення Docker...'
apt-get update -qq
apt-get install -y -qq ca-certificates curl gnupg
install -m 0755 -d /etc/apt/keyrings
curl -fsSL https://download.docker.com/linux/ubuntu/gpg | gpg --dearmor -o /etc/apt/keyrings/docker.gpg
chmod a+r /etc/apt/keyrings/docker.gpg
echo "deb [arch=$(dpkg --print-architecture) signed-by=/etc/apt/keyrings/docker.gpg] https://download.docker.com/linux/ubuntu $(. /etc/os-release && echo \"$VERSION_CODENAME\") stable" | tee /etc/apt/sources.list.d/docker.list > /dev/null
apt-get update -qq
apt-get install -y -qq docker-ce docker-ce-cli containerd.io docker-buildx-plugin docker-compose-plugin
service docker start
sleep 3

echo '📦 Встановлення K3d...'
curl -s https://raw.githubusercontent.com/k3d-io/k3d/main/install.sh | bash

echo '📦 Встановлення Helm...'
curl -fsSL -o get_helm.sh https://raw.githubusercontent.com/helm/helm/main/scripts/get-helm-3
bash get_helm.sh

echo '📦 Встановлення zrok...'
curl -sSLf https://get.openziti.io/install.bash | sudo bash -s zrok
/usr/local/bin/zrok version

echo '✅ Інфраструктура Colab готова!'

In [ ]:
%%bash
# 2. КЛОНУВАННЯ РЕПОЗИТОРІЮ
echo '🔄 Клонування репозиторію PREDATOR...'
rm -rf /opt/predator
git clone https://github.com/dima1203oleg/predator-analytics.git /opt/predator
cd /opt/predator
echo '✅ Репозиторій успішно завантажено!'

In [ ]:
%%bash
# 3. ЗБІРКА КОНТЕЙНЕРІВ (ТАК САМО ЯК НА IMAC)
cd /opt/predator
echo '🔨 Збірка Frontend... (це займе ~2-3 хвилини)'
docker build -t predator/frontend:v61.0-ELITE -f apps/predator-analytics-ui/Dockerfile .

echo '🔨 Збірка Core API...'
docker build -t predator/core-api:v61.0-ELITE -f services/core-api/Dockerfile services/core-api/

echo '🔨 Збірка Graph Service...'
docker build -t predator/graph-service:v61.0-ELITE -f services/graph-service/Dockerfile services/graph-service/

echo '🔨 Збірка Ingestion Worker...'
docker build -t predator/ingestion-worker:v61.0-ELITE -f services/ingestion-worker/Dockerfile services/ingestion-worker/

echo '✅ Всі образи зібрано!'

In [ ]:
%%bash
# 4. СТВОРЕННЯ K3D КЛАСТЕРА ТА ІМПОРТ ОБРАЗІВ
echo '🚀 Створення k3d кластера...'
k3d cluster delete predator-cluster 2>/dev/null || true
k3d cluster create predator-cluster \
    -p "8000:80@loadbalancer" \
    -p "3030:3030@loadbalancer" \
    --k3s-arg "--disable=traefik@server:0" \
    --agents 1

echo '📦 Імпорт образів у кластер...'
k3d image import predator/frontend:v61.0-ELITE predator/core-api:v61.0-ELITE predator/graph-service:v61.0-ELITE predator/ingestion-worker:v61.0-ELITE -c predator-cluster

echo '✅ Кластер розгорнуто!'

In [ ]:
%%bash
# 5. ДЕПЛОЙ HELM ЧАРТІВ
cd /opt/predator
echo '⚙️ Розгортання PREDATOR Helm Chart...'
helm upgrade --install predator ./deploy/helm/predator \
    --set frontend.image.tag=v61.0-ELITE \
    --set coreApi.image.tag=v61.0-ELITE \
    --set graphService.image.tag=v61.0-ELITE \
    --set ingestionWorker.image.tag=v61.0-ELITE \
    --set global.domain="localhost"

echo '⏳ Очікування готовності сервісів (може зайняти до 2 хвилин)...'
kubectl wait --for=condition=available --timeout=300s deployment/predator-frontend
kubectl wait --for=condition=available --timeout=300s deployment/predator-core-api
echo '✅ PREDATOR успішно розгорнуто в Kubernetes (Google Colab)!'

In [ ]:
# 6. ZROK ТУНЕЛЬ ДЛЯ ДОСТУПУ З MACBOOK
import os, subprocess, re, time

ZROK_TOKEN = '1eeje4um7yvA' # Автоматично підставлено з системи

print('🌐 Налаштування zrok...')
if ZROK_TOKEN != 'ВАШ_ZROK_TOKEN_ТУТ':
    subprocess.run(f'/usr/local/bin/zrok enable {ZROK_TOKEN}', shell=True)

print('🚀 Запуск zrok share public http://localhost:8000...')
zrok_proc = subprocess.Popen(
    ['/usr/local/bin/zrok', 'share', 'public', 'http://localhost:8000', '--headless'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    universal_newlines=True
)

public_url = None
for _ in range(60):
    line = zrok_proc.stdout.readline()
    if not line: break
    print(line.strip())
    match = re.search(r'https://[a-z0-9\-]+\.share\.zrok\.io', line)
    if match:
        public_url = match.group(0)
        break
    time.sleep(1)

if public_url:
    print('\n' + '='*65)
    print('🦅 PREDATOR Analytics — COLAB K8S CLUSTER АКТИВНИЙ (ZROK)')
    print('='*65)
    print(f'\n🌐  Публічний URL: {public_url}')
    print(f'⚙️   API:           {public_url}/api/v1')
    print('\n' + '─'*65)
    print('👇 ВСТАВТЕ У .env.local на вашому Mac:')
    print(f'VITE_API_URL={public_url}/api/v1')
    print(f'VITE_BACKEND_PROXY_TARGET={public_url}')
    print('VITE_MODE=cloud')
    print('─'*65)
else:
    print('❌ Не вдалося отримати zrok URL. Перевірте статус zrok.')


In [ ]:
# 7. ПІДТРИМКА СЕСІЇ (Keep-Alive)
import time, datetime
print('🦅 Кластер працює. Не закривайте цю клітинку.')
while True:
    print(f'\r⏱️ {datetime.datetime.now().strftime("%H:%M:%S")} | K3D Cluster Active', end='')
    time.sleep(60)